# Modèles de Base – LIAR Dataset

L'objectif de ce notebook est de construire des baselines solides in‑domain (sur le LIAR dataset) utilisant TF‑IDF couplé à `LogisticRegression` et `LinearSVC`.

Nous ciblons la classification binaire (`label_binary`) afin de nous aligner par la suite avec le système Fake/Real de BuzzFeed.
Ces modèles serviront de référence pour les évaluations ultérieures.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px

DATA_DIR = Path("../data")
TRAITEES_DIR = DATA_DIR / "traitees"
MODELS_DIR = DATA_DIR / "modeles"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Chargement des splits stratifiés figés lors de l'EDA
train = pd.read_parquet(TRAITEES_DIR / "liar_train.parquet")
val   = pd.read_parquet(TRAITEES_DIR / "liar_val.parquet")
test  = pd.read_parquet(TRAITEES_DIR / "liar_test.parquet")

print(f"Tailles - Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Tailles - Train: 10240, Val: 1284, Test: 1267


## 1. Préparation des variables cibles

In [2]:
X_train, y_train = train["statement"], train["label_binary"]
X_val, y_val = val["statement"], val["label_binary"]
X_test, y_test = test["statement"], test["label_binary"]

# Afin d'utiliser un GridSearchCV qui entraîne sur Train et valide explicitement sur Val :
from sklearn.model_selection import PredefinedSplit

X_train_val = pd.concat([X_train, X_val]).reset_index(drop=True)
y_train_val = pd.concat([y_train, y_val]).reset_index(drop=True)

# Indices: -1 pour les éléments à toujours laisser dans le train, 0 pour le split de validation
test_fold = np.concatenate([-1 * np.ones(len(X_train)), np.zeros(len(X_val))])
ps = PredefinedSplit(test_fold)

## 2. Pipeline TF-IDF + Logistic Regression

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

pipeline_lr = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=2, max_df=0.9)),
    ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
])

params_lr = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'clf__C': [0.1, 1.0, 10.0]
}

grid_lr = GridSearchCV(pipeline_lr, params_lr, cv=ps, scoring='f1_macro', n_jobs=-1, verbose=1)
grid_lr.fit(X_train_val, y_train_val)

best_lr = grid_lr.best_estimator_
print("Meilleurs paramètres LR:", grid_lr.best_params_)

Fitting 1 folds for each of 6 candidates, totalling 6 fits
Meilleurs paramètres LR: {'clf__C': 1.0, 'tfidf__ngram_range': (1, 2)}


In [4]:
y_pred_lr = best_lr.predict(X_test)
acc_lr = accuracy_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr, average='macro')

print(f"Logistic Regression sur Test - Accuracy: {acc_lr:.4f}, Macro-F1: {f1_lr:.4f}\n")
print(classification_report(y_test, y_pred_lr))

Logistic Regression sur Test - Accuracy: 0.6298, Macro-F1: 0.6249

              precision    recall  f1-score   support

           0       0.57      0.59      0.58       553
           1       0.68      0.66      0.67       714

    accuracy                           0.63      1267
   macro avg       0.62      0.63      0.62      1267
weighted avg       0.63      0.63      0.63      1267



In [5]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
fig = px.imshow(cm_lr, text_auto=True, title="Matrice de Confusion - Logistic Regression", 
                labels=dict(x="Prédictions", y="Vérité"),
                x=['Faux/Plutôt Faux', 'Vrai/Plutôt Vrai'], 
                y=['Faux/Plutôt Faux', 'Vrai/Plutôt Vrai'])
fig.show()

## 3. Pipeline TF-IDF + LinearSVC

In [6]:
from sklearn.svm import LinearSVC

pipeline_svc = Pipeline([
    ('tfidf', TfidfVectorizer(min_df=2, max_df=0.9)),
    ('clf', LinearSVC(class_weight='balanced', max_iter=2000, random_state=42))
])

params_svc = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'clf__C': [0.01, 0.1, 1.0, 10.0]
}

grid_svc = GridSearchCV(pipeline_svc, params_svc, cv=ps, scoring='f1_macro', n_jobs=-1, verbose=1)
grid_svc.fit(X_train_val, y_train_val)

best_svc = grid_svc.best_estimator_
print("Meilleurs paramètres SVC:", grid_svc.best_params_)

Fitting 1 folds for each of 8 candidates, totalling 8 fits
Meilleurs paramètres SVC: {'clf__C': 0.1, 'tfidf__ngram_range': (1, 2)}


In [7]:
y_pred_svc = best_svc.predict(X_test)
acc_svc = accuracy_score(y_test, y_pred_svc)
f1_svc = f1_score(y_test, y_pred_svc, average='macro')

print(f"LinearSVC sur Test - Accuracy: {acc_svc:.4f}, Macro-F1: {f1_svc:.4f}\n")
print(classification_report(y_test, y_pred_svc))

LinearSVC sur Test - Accuracy: 0.6314, Macro-F1: 0.6266

              precision    recall  f1-score   support

           0       0.58      0.59      0.58       553
           1       0.68      0.66      0.67       714

    accuracy                           0.63      1267
   macro avg       0.63      0.63      0.63      1267
weighted avg       0.63      0.63      0.63      1267



In [8]:
cm_svc = confusion_matrix(y_test, y_pred_svc)
fig2 = px.imshow(cm_svc, text_auto=True, title="Matrice de Confusion - LinearSVC", 
                 labels=dict(x="Prédictions", y="Vérité"),
                 x=['Faux/Plutôt Faux', 'Vrai/Plutôt Vrai'], 
                 y=['Faux/Plutôt Faux', 'Vrai/Plutôt Vrai'])
fig2.show()

## 4. Sauvegarde des modèles et métriques

Nous figeons les modèles ainsi que les scores de test afin de s'en servir de base de référence lors de l'évaluation hors de domaine.

In [9]:
import joblib
import json

# Sauvegarde Modèles
joblib.dump(best_lr, MODELS_DIR / "baseline_logreg.joblib")
joblib.dump(best_svc, MODELS_DIR / "baseline_linearsvc.joblib")

# Sauvegarde Métriques Test
metrics = {
    "LogisticRegression": {
        "accuracy": float(acc_lr),
        "macro_f1": float(f1_lr)
    },
    "LinearSVC": {
        "accuracy": float(acc_svc),
        "macro_f1": float(f1_svc)
    }
}

with open(MODELS_DIR / "baselines_metrics_test.json", "w") as f:
    json.dump(metrics, f, indent=4)

print(f"Modèles sauvegardés dans : {MODELS_DIR}")
print("Métriques :", json.dumps(metrics, indent=4))

Modèles sauvegardés dans : ..\data\modeles
Métriques : {
    "LogisticRegression": {
        "accuracy": 0.6298342541436464,
        "macro_f1": 0.6248528226175573
    },
    "LinearSVC": {
        "accuracy": 0.6314127861089187,
        "macro_f1": 0.6265893283468587
    }
}
